# 3단계 · 검토한 CosFace 이후 전체 기준선

같은 RUN_ID를 사용합니다. [실행과 결과 안내](HOW_TO_READ.md)

In [ ]:
# cell 1 : 설정 및 경로 확인
import json
import os
import sys
from pathlib import Path

from IPython.display import display
import pandas as pd

repo = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "fine-tune/modules").is_dir())
sys.path.insert(0, str(repo / "fine-tune"))
from modules.clean_training import final_artifact, recipe, run_directory, run_stage
from modules.clean_campaign import model_dir, origin_matrix, parent_validation, read_result, run_full, run_probe, validate_choices

DATA_ROOT = Path(os.environ.get("FACE_DATASET_ROOT", "F:/face_datasets"))
MODEL_ROOT = Path(os.environ.get("FACE_MODEL_ROOT", "F:/models"))
RUN_ID = "carbon_mfn1024_s8972_v2"
SEED = 8972
EXECUTE = False
RESUME = False
KEEP_RAW_RESULTS = False
PROTOCOL_SCOPE = "paper_table1"
DATASETS = ["tinyface", "scface", "survface"]
HEADS = ["cosface", "adaface", "magface", "arcface"]
REVIEWED_PROBE_DATASETS = ["tinyface", "survface"]
APPROVE_EXPANSION = False
run = run_directory(MODEL_ROOT, RUN_ID)
validate_choices(DATASETS, HEADS, PROTOCOL_SCOPE, "clean_parent")
display(pd.DataFrame([dict(dataset=ds, head=head,
    directory=model_dir(run, PROTOCOL_SCOPE, "clean_parent", ds, head))
    for head in HEADS for ds in DATASETS]))

In [ ]:
# cell 2 : 실행
if EXECUTE:
    results = run_full(DATA_ROOT, run, PROTOCOL_SCOPE, DATASETS, HEADS, approved=APPROVE_EXPANSION,
                       reviewed_datasets=REVIEWED_PROBE_DATASETS, seed=SEED,
                       keep_raw_results=KEEP_RAW_RESULTS, resume=RESUME)

In [ ]:
# cell 3 : 결과 확인
if EXECUTE:
    display(pd.DataFrame(results))
else:
    rows = []
    for head in HEADS:
        for dataset in DATASETS:
            folder = model_dir(run, PROTOCOL_SCOPE, "clean_parent", dataset, head)
            completed = (folder / "COMPLETED").is_file()
            snapshot = folder / ("summary.json" if completed else "progress.json")
            record = json.loads(snapshot.read_bytes()) if snapshot.is_file() else {}
            epochs = record.get("epochs", [])
            last_epoch = epochs[-1] if epochs else {}
            status = "completed" if completed else ("in_progress" if record else "pending")
            rows.append(dict(dataset=dataset, head=head, COMPLETED=completed, status=status,
                             epoch=record.get("epoch"), loss=last_epoch.get("loss"),
                             head_direction_norm=last_epoch.get("head_direction_norm")))
    display(pd.DataFrame(rows).astype({"epoch": "Int64"}))
    print("epoch과 지표는 마지막으로 저장된 학습 기록입니다.")
    print("02 결과 검토 후 APPROVE_EXPANSION을 설정합니다.")